# SatQuery AI — Step 7: BigEarthNet land-cover encoder

Trains a multi-label land-cover classifier on BigEarthNet in **three variants —
Sentinel-2-only, Sentinel-1-only, and fused** — so the resulting ablation table is
real evidence that fusing optical+SAR beats either sensor alone (SIH26167
capability #4: optical+SAR cross-modal analysis).

**Run this on Google Colab** with a GPU runtime on (Runtime → Change runtime
type → T4 GPU). The training budget below is deliberately small (a few
thousand patches, a handful of epochs) — per the build plan, a real number from a
modest subset is all the internal round (7 Sept 2026) needs. This does not aim to
reproduce published BigEarthNet benchmark numbers.

**Before running:** you need a Kaggle API token to pull the dataset
programmatically (Colab has no "Add Data" mount like Kaggle does):
1. On kaggle.com, go to **Settings → API → Create New Token** — this gives you
   a token starting with `KGAT_`.
2. Run the next cell and paste it into the hidden prompt (it's never written
   into this notebook's saved source, and the token is cleared from memory
   right after use).
3. Treat that token like a password: once you're done training, regenerate/
   revoke it from kaggle.com/settings if you've pasted it anywhere it could
   leak (a chat log, a shared screen, etc).

This notebook downloads
**[BigEarthNet 14K](https://www.kaggle.com/datasets/narendraaironi/bigearthnet-14k)**
by Narendra Aironi — ~13.7k S1+S2-paired patches, 5.54GB, CDLA-Permissive-1.0
license. Its exact layout (verified 2026-09-04) is what the discovery cell below
targets directly:

```
BEN_14k/
  BigEarthNet-S2/{train,test,validation}/<patch_id>.tif   (10 bands, pre-stacked)
  BigEarthNet-S1/{train,test,validation}/<s1_name>.tif    (2 bands: VV, VH)
  metadata.parquet   # patch_id, labels, split, country, s1_name, s2v1_name, ...
```

If you use a different BigEarthNet mirror instead, the discovery cell searches
for any `metadata.parquet` under the downloaded input folder with the right
columns rather than assuming this exact dataset — but a genuinely different
layout (e.g. the classic one-file-per-band + JSON-sidecar release) will need
`MANUAL_METADATA_PATH` and a small edit to the patch-index cell to match it.

**Output contract with the rest of the repo:** this notebook saves
`landcover_<variant>.pt` + `landcover_<variant>_classes.json` per variant to
`/content/models_weights/`, then zips and downloads that folder straight to your
machine via the browser at the end. `satquery/models/landcover.py` expects
those exact filenames — drop the unzipped folder at the repo root when you're
done and nothing else needs to change.


In [ ]:
%pip install -q rasterio scikit-learn tabulate pyarrow kaggle --no-warn-script-location


## Kaggle authentication + dataset download

Paste the `KGAT_...` token from kaggle.com/settings into the hidden prompt
below (via `getpass`, so it never appears in this notebook's saved source or
its output). It's used only to download this one dataset via the Kaggle CLI.


In [ ]:
import getpass
import os
from pathlib import Path

kaggle_token = getpass.getpass("Paste your Kaggle API token (starts with KGAT_): ")
os.environ["KAGGLE_API_TOKEN"] = kaggle_token

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
token_path = kaggle_dir / "access_token"
token_path.write_text(kaggle_token)
token_path.chmod(0o600)

del kaggle_token
print("Kaggle token set (env var + ~/.kaggle/access_token) for this session only.")


In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/content/input")
INPUT_ROOT.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d narendraaironi/bigearthnet-14k -p {INPUT_ROOT} --unzip


In [ ]:
import json
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet50

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Config

Tune `max_train_patches` / `epochs` up if Colab's session-time budget allows;
the defaults are sized to comfortably finish within a single Colab session on a
T4. `max_val_patches` caps eval time the same way.


In [ ]:
CONFIG = {
    "max_train_patches": 3000,   # a few thousand patches is enough for a real number (see build plan)
    "max_val_patches": 800,
    "batch_size": 32,
    "epochs": 6,
    "lr": 3e-4,
    "patch_size": 120,           # BigEarthNet's native patch size
    "num_workers": 2,
    "exclude_snow_cloud": True,  # drop patches flagged contains_seasonal_snow / contains_cloud_or_shadow, if present
    "out_dir": Path("/content/models_weights"),
}
CONFIG["out_dir"].mkdir(parents=True, exist_ok=True)
CONFIG


## Locate the downloaded BigEarthNet dataset

Searches the downloaded input folder (breadth-limited, so this stays fast even
on a large dataset) for a `metadata.parquet` carrying the columns this notebook
needs: `patch_id`, `labels`, `split`, `s1_name`. Once found, the Sentinel-2 and
Sentinel-1 patch folders are assumed to sit alongside it as
`BigEarthNet-S2/<split>/<patch_id>.tif` and `BigEarthNet-S1/<split>/<s1_name>.tif`
— exactly how the *BigEarthNet 14K* dataset linked above is laid out.


In [ ]:
MANUAL_METADATA_PATH: str | None = None  # set to an exact metadata.parquet path if auto-discovery below finds nothing


def tree_summary(root: Path, max_entries: int = 30) -> None:
    if not root.exists():
        print(f"{root} does not exist (download cell above didn't run?)")
        return
    print(f"first {max_entries} entries under {root} (breadth-limited scan):")
    count = 0
    for dirpath, dirnames, filenames in os.walk(root):
        for name in dirnames + filenames:
            print(" ", Path(dirpath, name).relative_to(root))
            count += 1
            if count >= max_entries:
                return


tree_summary(INPUT_ROOT)


In [ ]:
REQUIRED_COLUMNS = {"patch_id", "labels", "split", "s1_name"}


def find_metadata_parquet(root: Path, max_files_checked: int = 20) -> Path | None:
    checked = 0
    for dirpath, _, filenames in os.walk(root):
        for name in filenames:
            if not name.endswith(".parquet"):
                continue
            path = Path(dirpath, name)
            checked += 1
            try:
                cols = set(pd.read_parquet(path).columns)
            except Exception as exc:
                print(f"  (skipping {path}: {exc})")
                continue
            if REQUIRED_COLUMNS.issubset(cols):
                return path
            if checked >= max_files_checked:
                return None
    return None


metadata_path = Path(MANUAL_METADATA_PATH) if MANUAL_METADATA_PATH else find_metadata_parquet(INPUT_ROOT)

if metadata_path is None:
    raise RuntimeError(
        "No metadata.parquet with columns {patch_id, labels, split, s1_name} found under "
        f"{INPUT_ROOT}. Check the download cell above ran without error, or set "
        "MANUAL_METADATA_PATH above to the exact path, then re-run from this cell."
    )

DATASET_ROOT = metadata_path.parent
S2_ROOT = DATASET_ROOT / "BigEarthNet-S2"
S1_ROOT = DATASET_ROOT / "BigEarthNet-S1"
print(f"metadata: {metadata_path}")
print(f"S2 root:  {S2_ROOT}  (exists: {S2_ROOT.exists()})")
print(f"S1 root:  {S1_ROOT}  (exists: {S1_ROOT.exists()})")


## Build the patch index

`metadata.parquet` isn't scoped to just this download — it's the reference
table for the *entire* BigEarthNet-v2 archive (480k+ patches), while this
"14K" dataset only actually ships the ~13.7k files that landed on disk under
`BigEarthNet-S2/`/`BigEarthNet-S1/`. Sampling from the metadata table first and
checking file existence after (the obvious approach) mostly picks rows whose
files were never downloaded — so this goes the other way: **list the files
that actually exist on disk first**, then look up their labels in the metadata
table, so the sample is never wasted on patches that aren't there.

Training and validation patches are sampled **per the dataset's own `split`
column**, not a fresh random split, since that's the split the dataset's
authors intended.

`NUM_CLASSES` reports exactly how many distinct labels were observed in the
sampled rows — this notebook doesn't assume the 19-class nomenclature, it just
happens to match here (confirmed from sample rows: "Arable land", "Broad-leaved
forest", "Mixed forest", ...). Whatever it is, the exact class list is saved
alongside the trained weights so inference always uses the model's real classes.


In [ ]:
def parse_labels(value) -> list[str]:
    if isinstance(value, (list, np.ndarray)):
        return [str(v) for v in value]
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return [str(v) for v in parsed] if isinstance(parsed, list) else []
        except json.JSONDecodeError:
            return []
    return []


def patch_ids_on_disk(root: Path, split: str) -> set[str]:
    split_dir = root / split
    if not split_dir.exists():
        return set()
    return {p.stem for p in split_dir.glob("*.tif")}


s2_train_ids = patch_ids_on_disk(S2_ROOT, "train")
s2_val_ids = patch_ids_on_disk(S2_ROOT, "validation")
s1_all_names = {name for split in ("train", "validation", "test") for name in patch_ids_on_disk(S1_ROOT, split)}
print(f"files on disk: {len(s2_train_ids)} S2 train, {len(s2_val_ids)} S2 validation, {len(s1_all_names)} S1 (any split)")

table = pd.read_parquet(metadata_path)
print(f"metadata.parquet: {len(table)} total row(s) (covers the full BigEarthNet-v2 archive, not just this download)")

if CONFIG["exclude_snow_cloud"]:
    for flag_col in ("contains_seasonal_snow", "contains_cloud_or_shadow"):
        if flag_col in table.columns:
            before = len(table)
            table = table[~table[flag_col].astype(bool)]
            print(f"  dropped {before - len(table)} row(s) flagged {flag_col}")

table["labels_parsed"] = table["labels"].apply(parse_labels)
table = table[table["labels_parsed"].map(len) > 0]

# restrict to rows whose S2 file is actually present on disk for that split -
# this is the fix: sample from what's downloaded, not from the full archive
train_rows = table[(table["split"] == "train") & (table["patch_id"].isin(s2_train_ids))]
val_rows = table[(table["split"] == "validation") & (table["patch_id"].isin(s2_val_ids))]
print(f"metadata rows matching files on disk: {len(train_rows)} train, {len(val_rows)} validation")

train_sample = train_rows.sample(n=min(CONFIG["max_train_patches"], len(train_rows)), random_state=SEED)
val_sample = val_rows.sample(n=min(CONFIG["max_val_patches"], len(val_rows)), random_state=SEED)
print(f"sampled: {len(train_sample)} train, {len(val_sample)} validation")

all_classes = sorted({label for labels in table["labels_parsed"] for label in labels})
class_to_idx = {c: i for i, c in enumerate(all_classes)}
NUM_CLASSES = len(all_classes)
print(f"observed {NUM_CLASSES} distinct label(s)")


def build_patch_records(rows: pd.DataFrame) -> list[dict]:
    records = []
    for _, row in rows.iterrows():
        s2_path = S2_ROOT / row["split"] / f"{row['patch_id']}.tif"
        s1_name = row.get("s1_name")
        has_s1 = isinstance(s1_name, str) and s1_name in s1_all_names
        s1_path = (S1_ROOT / row["split"] / f"{s1_name}.tif") if has_s1 else None
        if s1_path is not None and not s1_path.exists():
            s1_path = None
        records.append({"s2_path": s2_path, "s1_path": s1_path, "labels": row["labels_parsed"]})
    return records


train_patches = build_patch_records(train_sample)
val_patches = build_patch_records(val_sample)
print(f"indexed: {len(train_patches)} train, {len(val_patches)} validation patch(es)")

s1_train_count = sum(1 for p in train_patches if p["s1_path"] is not None)
print(f"{s1_train_count}/{len(train_patches)} training patch(es) have a Sentinel-1 partner on disk")

VARIANTS = ["s2"]
if s1_train_count >= max(200, int(0.3 * max(len(train_patches), 1))):
    VARIANTS += ["s1", "fused"]
    print("enough S1-paired patches found - training s2, s1, and fused variants")
else:
    print("not enough S1-paired patches - training s2-only")


## Dataset and model

Each patch file is already a pre-stacked multi-band GeoTIFF (10 bands for S2, 2
for S1) — no per-band file lookup needed, just read the whole file.

**Every needed file is read from disk exactly once, up front, into an in-memory
cache** (`ARRAY_CACHE`), rather than inside `Dataset.__getitem__` — with only a
few thousand unique patches at ~0.5MB each (well under Colab's free-tier RAM),
reading each file fresh on *every* epoch of *every* variant is pure waste: the
training loop below reuses the same files across 6 epochs and up to 3 variants,
so caching turns N-epochs-times-N-variants disk reads into exactly 1 per file.

12-channel (10 S2 + 2 S1) stem-widened ResNet-50, ImageNet-initialised: the
pretrained RGB-ish filters seed the first 3 input channels, and every extra
spectral/radar channel starts from the mean of those filters rather than random
noise. Matches `satquery/models/landcover.py`'s `_load_model` exactly, so a
checkpoint saved here loads there with zero changes.


In [ ]:
IN_CHANNELS = {"s2": 10, "s1": 2, "fused": 12}


def read_patch(path: Path, size: int) -> np.ndarray:
    with rasterio.open(path) as ds:
        arr = ds.read().astype(np.float32)
    if arr.shape[-2:] != (size, size):
        t = torch.from_numpy(arr).unsqueeze(0)
        arr = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)[0].numpy()
    return arr


def normalize(arr: np.ndarray) -> np.ndarray:
    lo = arr.min(axis=(1, 2), keepdims=True)
    hi = arr.max(axis=(1, 2), keepdims=True)
    return (arr - lo) / np.clip(hi - lo, 1e-6, None)


def preload(paths: set, size: int) -> dict:
    cache = {}
    total = len(paths)
    for i, path in enumerate(paths):
        cache[path] = read_patch(path, size)
        if (i + 1) % 500 == 0 or (i + 1) == total:
            print(f"  preloaded {i + 1}/{total} file(s)")
    return cache


_needed_paths = set()
for _rec in train_patches + val_patches:
    _needed_paths.add(_rec["s2_path"])
    if _rec["s1_path"] is not None:
        _needed_paths.add(_rec["s1_path"])

print(f"preloading {len(_needed_paths)} unique patch file(s) into memory (read once, reused across every epoch/variant)...")
ARRAY_CACHE = preload(_needed_paths, CONFIG["patch_size"])
print("preload done")


class BigEarthNetPatchDataset(Dataset):
    def __init__(self, records: list[dict], variant: str):
        self.records = records
        self.variant = variant

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        if self.variant == "s2":
            arr = ARRAY_CACHE[rec["s2_path"]]
        elif self.variant == "s1":
            arr = ARRAY_CACHE[rec["s1_path"]]
        else:  # fused
            arr = np.concatenate([ARRAY_CACHE[rec["s2_path"]], ARRAY_CACHE[rec["s1_path"]]], axis=0)

        arr = normalize(arr)
        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        for label in rec["labels"]:
            target[class_to_idx[label]] = 1.0
        return torch.from_numpy(arr), torch.from_numpy(target)


def build_model(in_channels: int, num_classes: int) -> nn.Module:
    model = resnet50(weights="IMAGENET1K_V2")
    old_conv1 = model.conv1
    new_conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    with torch.no_grad():
        k = min(3, in_channels)
        new_conv1.weight[:, :k] = old_conv1.weight[:, :k]
        if in_channels > 3:
            mean_filter = old_conv1.weight.mean(dim=1, keepdim=True)
            new_conv1.weight[:, 3:] = mean_filter.repeat(1, in_channels - 3, 1, 1)
    model.conv1 = new_conv1
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)


## Train + evaluate each variant

Standard multi-label setup: `BCEWithLogitsLoss`, sigmoid + 0.5 threshold at eval
time, micro/macro-F1 (the two metrics BigEarthNet papers report). Each variant's
weights and the exact class list it was trained on are saved as soon as training
finishes, so a crash on a later variant doesn't lose earlier results.


In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, n = 0.0, 0
    all_targets, all_probs = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(is_train):
            logits = model(x)
            loss = F.binary_cross_entropy_with_logits(logits, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
        all_targets.append(y.detach().cpu().numpy())
        all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
    return total_loss / max(n, 1), np.concatenate(all_targets), np.concatenate(all_probs)


def train_variant(variant: str) -> dict:
    print(f"\n=== training variant: {variant} ===")
    train_records = [p for p in train_patches if (variant == "s2" or p["s1_path"] is not None)]
    val_records = [p for p in val_patches if (variant == "s2" or p["s1_path"] is not None)]

    train_loader = DataLoader(
        BigEarthNetPatchDataset(train_records, variant),
        batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"],
    )
    val_loader = DataLoader(
        BigEarthNetPatchDataset(val_records, variant),
        batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"],
    )

    model = build_model(IN_CHANNELS[variant], NUM_CLASSES)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])

    micro_f1 = macro_f1 = 0.0
    for epoch in range(CONFIG["epochs"]):
        train_loss, _, _ = run_epoch(model, train_loader, optimizer)
        val_loss, val_targets, val_probs = run_epoch(model, val_loader, optimizer=None)
        val_preds = (val_probs >= 0.5).astype(int)
        micro_f1 = f1_score(val_targets, val_preds, average="micro", zero_division=0)
        macro_f1 = f1_score(val_targets, val_preds, average="macro", zero_division=0)
        print(
            f"epoch {epoch + 1}/{CONFIG['epochs']}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
            f"micro_f1={micro_f1:.3f}  macro_f1={macro_f1:.3f}"
        )

    weights_path = CONFIG["out_dir"] / f"landcover_{variant}.pt"
    classes_path = CONFIG["out_dir"] / f"landcover_{variant}_classes.json"
    torch.save(model.state_dict(), weights_path)
    classes_path.write_text(json.dumps(all_classes, indent=2))
    print(f"saved {weights_path.name} and {classes_path.name}")

    return {
        "variant": variant,
        "num_train": len(train_records),
        "num_val": len(val_records),
        "num_classes": NUM_CLASSES,
        "micro_f1": round(float(micro_f1), 3),
        "macro_f1": round(float(macro_f1), 3),
    }


results = [train_variant(v) for v in VARIANTS]


## The ablation table

This is what goes in the PPT — a three-row table (or one, if no Sentinel-1
partner was resolvable this run) where the fused variant's micro-F1 should beat
both single-sensor baselines. That comparison is the evidence for the
optical+SAR requirement, not just an assertion.


In [ ]:
ablation = pd.DataFrame(results)[["variant", "num_train", "num_val", "num_classes", "micro_f1", "macro_f1"]]
ablation_path = CONFIG["out_dir"] / "ablation_table.csv"
ablation.to_csv(ablation_path, index=False)

print(ablation.to_markdown(index=False))
print(f"\nsaved {ablation_path}")

if "fused" in ablation["variant"].values:
    fused_micro = ablation.loc[ablation.variant == "fused", "micro_f1"].iloc[0]
    best_single = ablation.loc[ablation.variant != "fused", "micro_f1"].max()
    verdict = "beats" if fused_micro > best_single else "does NOT beat"
    print(f"\nfused micro-F1 ({fused_micro:.3f}) {verdict} the best single-sensor micro-F1 ({best_single:.3f})")
else:
    print("\nonly s2 trained this run - not enough resolvable Sentinel-1 partners for the s1/fused rows")

ablation


In [ ]:
import shutil

from google.colab import files as colab_files

zip_base = "/content/models_weights"
shutil.make_archive(zip_base, "zip", CONFIG["out_dir"])
print(f"zipped {CONFIG['out_dir']} -> {zip_base}.zip - starting download")
colab_files.download(f"{zip_base}.zip")


## Getting the weights back into SatQuery AI

1. The cell above should have triggered a browser download of
   `models_weights.zip` automatically. If your browser blocked the popup, open
   Colab's file browser (folder icon, left sidebar) and download
   `/content/models_weights.zip` manually.
2. Unzip it and copy the folder to the repo root, so the path is
   `C:\Users\Ariha\OneDrive\Desktop\Sih-167\models_weights\landcover_<variant>.pt`
   (plus the matching `_classes.json` files, and `ablation_table.csv` for the PPT).
3. That's it — `satquery/models/landcover.py` picks the weights up automatically
   the next time a `vqa`/`captioning` question runs against a band-matching image.
   No code changes needed anywhere else in the pipeline.
4. Locally, `pip install torch torchvision` inside `.venv` if you haven't already
   — inference at demo time is a single image on CPU, which is fine on this
   laptop even though training itself had to happen on Colab's GPU.
5. To actually *see* a real prediction in the browser demo, the uploaded image
   needs to match a trained variant's band count exactly (10 for s2, 2 for s1,
   12 for fused) — the repo's existing synthetic fixtures are 12-band and won't
   trigger the s2-only model. Use a real patch from this dataset (or ask for a
   matching synthetic fixture to be generated) to see it live.
